Repo URL: https://github.com/<redacted>/assignment-4-equation-of-a-slime-<redacted>
Commit ID: 3769425f9d19e3c261312547d8047cb66662b689

In [13]:
# Imports section
import pandas as pd
import numpy as np

## Part 1. Loading the dataset

In [14]:
# Using pandas load the dataset (load remotely, not locally) 

# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)

df = pd.read_csv("science_data_large.csv")
print(df.head(15))
print(f"\nThere are {df.shape[0]} rows and {df.shape[1]} columns in the science data table.\nThe following are the column labels of the table: {df.columns}")
df.info

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04

There are 1000 rows and 3 columns in the science data table.
The following are the column labels of the table: Index(['Temperature °C', 'Mols KCL', 'Size nm^3'], dtype='object')


<bound method DataFrame.info of      Temperature °C  Mols KCL     Size nm^3
0               469       647  6.244743e+05
1               403       694  5.779610e+05
2               302       975  6.196847e+05
3               779       916  1.460449e+06
4               901        18  4.325726e+04
..              ...       ...           ...
995             894       847  1.545661e+06
996             327       982  6.737041e+05
997             791       213  3.477543e+05
998             769       553  8.684794e+05
999             919       452  8.476413e+05

[1000 rows x 3 columns]>

## Part 2. Splitting the dataset

In [15]:
from sklearn.model_selection import train_test_split

# Take the pandas dataset and split it into our features (X) and label (y)
features = df.iloc[:, :-1] 
label = df.iloc[:, -1]

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(features, label, random_state=42, train_size=0.9)



## Part 3. Perform a Linear Regression

In [16]:
# Use sklearn to train a model on the training set
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"The r-squared score for the regression model is {round(r2, 4)}")

# Create a sample datapoint and predict the output of that sample with the trained model
data = [250, 750]
sample_data = pd.DataFrame(np.array(data).reshape(1, -1), columns=['Temperature °C', 'Mols KCL'])
sample_pred = model.predict(sample_data)
print(f"Sample data: {data}\nThis is the prediction for the size of the sample data point: {sample_pred[0]:.6e}")

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
# In markdown cell below...

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
coefficents = model.coef_
intercept = model.intercept_

coefficents, intercept

The r-squared score for the regression model is 0.8552
Sample data: [250, 750]
This is the prediction for the size of the sample data point: 5.816664e+05


(array([ 866.14641337, 1032.69506649]), -409391.47958340764)

Linear Regression Equation: $f(T, M) = 866.14641337*T + 1032.69506649*M -409391.47958340764 $

### Report on the score for that model, in your own words (markdown, not code) explain what the score means

The evaluation score I used for the model is r-squared score and I got a value of 0.8552. R-squared value is calculated by 1 - SSR (Sum Squared Residuals)The residual of a point represents how far off the prediction was from the true value. Therefore the higher residuals there are, the less accurate the model is. Since r-squared is is 1 - SSR, the lower the SSR or residuals -> higher the r-squared score. This means we are aiming for an r-squared as close to 1 as possible, since 1 is its maximum value.
<br>The model above reports an r-squared score of around 0.8552. This is quite mediocre, while it may seem somewhat close to 1. 0.855 means there is still a significant amount of error. I believe this may be due to an insufficient amount of training data since only 900 data points were utilized. To get a more accurate model, we require much larger datasets.

## Part 4. Use Cross Validation

In [17]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, features, label, cv=kf)
mean = cv_scores.mean()
std = cv_scores.std()

print(cv_scores)
print(f'mean: {round(mean, 4)} std: {round(std, 4)}')

# Report on their finding and their significance
# In below markdown cell...

[0.85524721 0.86563495 0.84842028 0.80043036 0.87514932 0.86941354
 0.89079607 0.87174054 0.8607738  0.85112286]
mean: 0.8589 std: 0.0228


### Report on their finding and their significance

Currently, I have n_splits set to 10, so the experiment is repeated across 10 shuffles of data. This resulted in scores ranging from 0.8 - 0.89
with a standard deviation of 0.0228, which is quite consistent. This means the model is pretty robust and likely does well with unseen data. 
If the number of splits is increased, it will cause much more variability in the models performance on each subset of data, and vice-versa when
the number of splits decreases. In general, the model is performing consistently, which is good to know for accuracy purposes.

## Part 5. Using Polynomial Regression

In [18]:
from sklearn.preprocessing import PolynomialFeatures

# Step 1: Create polynomial features (degree 2)
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)  

# Step 2: Fit a linear regression model on the polynomial features
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)

# Step 3: Transform the test data
X_test_poly = poly.transform(X_test)  

# Step 4: Make predictions
y_pred = model_poly.predict(X_test_poly)

# Step 5: Evaluate the model
r2 = r2_score(y_test, y_pred)

print(f'R² Score: {r2}')

# Step 6: Output the coefficients to formulate the polynomial equation
coefficients_poly = model_poly.coef_
intercept_poly = model_poly.intercept_

print(f"Intercept: {intercept_poly}")
terms = poly.get_feature_names_out()  
for coef, term in zip(coefficients_poly, terms):
    print(f"Term: {term}, Coefficient: {coef}")

# Report on the metrics and output the resultant equation as you did in Part 3.
# In markdown cell below...

from sklearn.pipeline import Pipeline

print(f"\nHere are the results from cross-validation of the Polynomial Feature equation:")
degree = 2
pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=degree)),  
    ('model_poly', LinearRegression())                    
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, features, label, cv=kf)
mean = cv_scores.mean()
std = cv_scores.std()

print(cv_scores)
print(f'mean: {round(mean, 4)} std: {round(std, 4)}')

R² Score: 1.0
Intercept: 2.0479375962167978e-05
Term: 1, Coefficient: 0.0
Term: Temperature °C, Coefficient: 11.999999977691196
Term: Mols KCL, Coefficient: -1.2719708848869804e-07
Term: Temperature °C^2, Coefficient: 1.2645884339690383e-11
Term: Temperature °C Mols KCL, Coefficient: 2.0000000000172227
Term: Mols KCL^2, Coefficient: 0.028571428708642932

Here are the results from cross-validation of the Polynomial Feature equation:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
mean: 1.0 std: 0.0


### Report on the metrics and output the resultant equation as you did in Part 3.

Using the polynomial features the regression model was able to achieve an R-squared value of 1. This means that when comparing the results of the trained model on the test data it was able to practically perfectly predict all of the true values. This is an amazing feat and also had me a bit shocked for the model performance to jump from an R-squared score of approximately 0.86 with linear features compared to 1.0 with polynomial features. This is why to verify I conducted a cross validation experiment with the polynomial features model and also ended with an value of 1.0 on each trial. Since we are evaluating the model on a separate test data set, we know that it is not overfitting to the data we gave it to train on. Therefore, the polynomial features model is highly successful.

Polynomial feature equation: $f(T, M) = (2.048e-05) + (12)T + (-1.27e-11)M + (2)TM + (1.26e-11)T^2 + (0.0286)M^2 $